In [2]:
import sys, os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

def load_data():
    X, y = load_diabetes(return_X_y=True)
    X = (X - X.mean(0)) / X.std(0)
    y = y.reshape(-1, 1)
    y = (y - y.mean()) / y.std()

    return train_test_split(
        torch.FloatTensor(X),
        torch.FloatTensor(y),
        test_size=0.2,
        random_state=42
    )

def build():
    return nn.Sequential(
        nn.Linear(10, 64),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

def train(model, X, y):
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

    model.train()
    for _ in range(600):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

def eval(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).cpu().numpy()
    y_np = y.cpu().numpy()
    return {
        "mse": mean_squared_error(y_np, pred),
        "r2": r2_score(y_np, pred),
    }

def save(model, metrics):
    os.makedirs("outputFile", exist_ok=True)
    torch.save(model.state_dict(), "outputFile/diabetes_mlp.pt")
    with open("outputFile/diabetes_metrics.txt", "w") as f:
        f.write(str(metrics))

if __name__ == "__main__":
    X_train, X_val, y_train, y_val = load_data()
    model = build()

    train(model, X_train, y_train)

    val_metrics = eval(model, X_val, y_val)
    save(model, val_metrics)

    print(val_metrics)

    success = val_metrics["r2"] > 0.30
    if success:
        print("pass")
        sys.exit(0)
    else:
        print("fail")
        sys.exit(1)

{'mse': 0.5890568494796753, 'r2': 0.3407060503959656}
pass


SystemExit: 0

/Users/Varun/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
